<a href="https://colab.research.google.com/github/Realosunboy6/ISYE_NOTE_SPRING2026/blob/main/ISYE671_IP_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ISYE 671: Linear Optimization and Network Flows
# Exercises: Integer Programming — Modeling and Solving

**Instructor:** Dr. Ziteng Wang | **Date:** February 16, 2026

---

**Instructions:** For each problem, write complete AMPL models using the `.mod` + `.dat` file approach, solve with Gurobi, and answer all discussion questions.

---


## Setup


In [1]:
# Install AMPL Python API
!pip install -q amplpy


In [2]:
from amplpy import AMPL, ampl_notebook
import time
import matplotlib.pyplot as plt
import numpy as np

# Initialize AMPL with Gurobi solver
ampl = ampl_notebook(
    modules=["gurobi"],
    license_uuid="40a0a160-668d-492d-be4a-236079be8ff7"
)

print("AMPL environment ready with Gurobi!")


Licensed to Bundle #7384.7942 expiring 20260531: ISYE 671 linear optimization and network flows, Prof. Ziteng Wang, Northern Illinois University.
AMPL environment ready with Gurobi!




## Problem 1: LP Relaxation and Rounding (Capital Budgeting)

A venture capital firm must decide which of 8 startup proposals to fund. Each startup requires an investment across 3 funding rounds. The firm has limited capital in each round.

| Startup | Round 1 (\$M) | Round 2 (\$M) | Round 3 (\$M) | Expected Return (\$M) |
|---------|-------------|-------------|-------------|---------------------|
| S1      | 3           | 2           | 0           | 12                  |
| S2      | 1           | 4           | 3           | 15                  |
| S3      | 5           | 0           | 2           | 18                  |
| S4      | 2           | 3           | 1           | 9                   |
| S5      | 0           | 5           | 4           | 20                  |
| S6      | 4           | 1           | 3           | 14                  |
| S7      | 2           | 2           | 5           | 16                  |
| S8      | 6           | 0           | 0           | 10                  |

**Budget available:** Round 1: \$10M, Round 2: \$9M, Round 3: \$8M

**Additional constraints:**
- S1 and S3 are competing in the same market — fund at most one.
- S7 requires S2's technology — if S7 is funded, S2 must be funded too.

### Tasks

**(a)** Formulate this as a binary IP. Write the `.mod` and `.dat` files and solve with Gurobi.

**(b)** Solve the LP relaxation (change binary variables to continuous $[0,1]$). Which startups have fractional values? What is the LP optimal return?

**(c)** Round the LP relaxation solution: set all $x_j \geq 0.5$ to 1 and all $x_j < 0.5$ to 0. Is the rounded solution feasible? What is its objective value compared to the true IP optimal?

**(d)** Compute the integrality gap. Is this a tight or loose relaxation?


In [3]:
# =============================================
# PROBLEM 1: Your .mod file
# =============================================

%%writefile vc_funding.mod
# Venture Capital Funding Model

set STARTUPS;
set ROUNDS;

param cost   {STARTUPS, ROUNDS} >= 0;   # investment required per round
param ret    {STARTUPS}         >= 0;   # expected return
param budget {ROUNDS}           >= 0;   # available capital per round

var x {STARTUPS} binary;                # 1 = fund this startup

maximize TotalReturn:
    sum {j in STARTUPS} ret[j] * x[j];

subject to BudgetRound {r in ROUNDS}:
    sum {j in STARTUPS} cost[j,r] * x[j] <= budget[r];

# S1 and S3 compete in the same market — fund at most one
subject to CompetingMarket:
    x['S1'] + x['S3'] <= 1;

# S7 requires S2's technology
subject to TechDependency:
    x['S7'] <= x['S2'];


Overwriting vc_funding.mod


In [4]:
# =============================================
# PROBLEM 1: Your .dat file
# =============================================

%%writefile vc_funding.dat
# Data for VC Funding Model

set STARTUPS := S1 S2 S3 S4 S5 S6 S7 S8;
set ROUNDS   := R1 R2 R3;

# cost[startup, round]  (investment $M)
param cost:
       R1  R2  R3 :=
  S1    3   2   0
  S2    1   4   3
  S3    5   0   2
  S4    2   3   1
  S5    0   5   4
  S6    4   1   3
  S7    2   2   5
  S8    6   0   0 ;

param ret :=
  S1 12
  S2 15
  S3 18
  S4  9
  S5 20
  S6 14
  S7 16
  S8 10 ;

param budget :=
  R1 10
  R2  9
  R3  8 ;


Overwriting vc_funding.dat


In [5]:
# =============================================
# PROBLEM 1(a): Solve IP
# =============================================

vc_ip = AMPL()
vc_ip.read('vc_funding.mod')
vc_ip.read_data('vc_funding.dat')
vc_ip.option['solver'] = 'gurobi'
vc_ip.solve()

z_IP = vc_ip.get_objective('TotalReturn').value()
x_ip = {j: vc_ip.get_variable('x')[j].value()
        for j in vc_ip.get_set('STARTUPS').members()}

print('=== IP Solution ===')
print(f'Optimal return: ${z_IP}M')
print('Funded startups:', [j for j, v in sorted(x_ip.items()) if v > 0.5])


Gurobi 13.0.0: Gurobi 13.0.0: optimal solution; objective 47
57 simplex iterations
3 branching nodes
=== IP Solution ===
Optimal return: $47.0M
Funded startups: ['S2', 'S3', 'S6']


In [7]:
# =============================================
# PROBLEM 1(b): Solve LP relaxation
# =============================================

vc_lp = AMPL()
vc_lp.read('vc_funding.mod')
vc_lp.read_data('vc_funding.dat')

# Hint: use "redeclare var x {STARTUPS} >= 0, <= 1;" to relax
vc_lp.eval('redeclare var x {STARTUPS} >= 0, <= 1;')

vc_lp.option['solver'] = 'gurobi'
vc_lp.solve()

z_LP = vc_lp.get_objective('TotalReturn').value()
x_lp = {j: vc_lp.get_variable('x')[j].value()
        for j in vc_lp.get_set('STARTUPS').members()}

print('=== LP Relaxation Solution ===')
print(f'LP optimal return: ${z_LP:.4f}M')
print('\nVariable values:')
for j, v in sorted(x_lp.items()):
    flag = '  <-- FRACTIONAL' if 0.001 < v < 0.999 else ''
    print(f'  x[{j}] = {v:.4f}{flag}')

fractional = [j for j, v in x_lp.items() if 0.001 < v < 0.999]
print(f'\nFractional variables: {fractional}')


Gurobi 13.0.0: Gurobi 13.0.0: optimal solution; objective 57
5 simplex iterations
=== LP Relaxation Solution ===
LP optimal return: $57.0000M

Variable values:
  x[S1] = 0.2857  <-- FRACTIONAL
  x[S2] = 0.8571  <-- FRACTIONAL
  x[S3] = 0.7143  <-- FRACTIONAL
  x[S4] = 0.0000
  x[S5] = 1.0000
  x[S6] = 0.0000
  x[S7] = 0.0000
  x[S8] = 0.7857  <-- FRACTIONAL

Fractional variables: ['S1', 'S2', 'S3', 'S8']


In [8]:
# =============================================
# PROBLEM 1(c): Round and check feasibility
# =============================================

# After solving the LP relaxation, round each variable:
# For each startup, if LP value >= 0.5, set to 1; else set to 0
x_rounded = {j: (1 if v >= 0.5 else 0) for j, v in x_lp.items()}
print('Rounded decisions:', {j: v for j, v in sorted(x_rounded.items()) if v == 1})

# Data for manual feasibility check
cost_data = {
    'S1': [3,2,0], 'S2': [1,4,3], 'S3': [5,0,2], 'S4': [2,3,1],
    'S5': [0,5,4], 'S6': [4,1,3], 'S7': [2,2,5], 'S8': [6,0,0]
}
ret_data = {'S1':12,'S2':15,'S3':18,'S4':9,'S5':20,'S6':14,'S7':16,'S8':10}
budget_data = [10, 9, 8]

feasible = True
print('\nBudget usage check:')
for r in range(3):
    usage = sum(cost_data[j][r] * x_rounded[j] for j in x_rounded)
    ok = (usage <= budget_data[r])
    if not ok: feasible = False
    print(f'  Round {r+1}: used={usage}, limit={budget_data[r]}, {"OK" if ok else "VIOLATED"}')

s13 = x_rounded['S1'] + x_rounded['S3']
ok_s13 = (s13 <= 1)
if not ok_s13: feasible = False
print(f'  S1+S3 = {s13} <= 1: {"OK" if ok_s13 else "VIOLATED"}')

ok_dep = (x_rounded['S7'] <= x_rounded['S2'])
if not ok_dep: feasible = False
print(f'  x[S7]={x_rounded["S7"]} <= x[S2]={x_rounded["S2"]}: {"OK" if ok_dep else "VIOLATED"}')

obj_rounded = sum(ret_data[j] * x_rounded[j] for j in x_rounded)
print(f'\nRounded solution feasible: {feasible}')
print(f'Rounded objective:    ${obj_rounded}M')
print(f'True IP optimal:      ${z_IP}M')
print(f'Loss from rounding:   ${z_IP - obj_rounded}M')


Rounded decisions: {'S2': 1, 'S3': 1, 'S5': 1, 'S8': 1}

Budget usage check:
  Round 1: used=12, limit=10, VIOLATED
  Round 2: used=9, limit=9, OK
  Round 3: used=9, limit=8, VIOLATED
  S1+S3 = 1 <= 1: OK
  x[S7]=0 <= x[S2]=1: OK

Rounded solution feasible: False
Rounded objective:    $63M
True IP optimal:      $47.0M
Loss from rounding:   $-16.0M


In [9]:
# =============================================
# PROBLEM 1(d): Compute integrality gap
# =============================================

# Maximization: LP >= IP, so gap = (z_LP - z_IP) / z_IP * 100%
gap = (z_LP - z_IP) / z_IP * 100

print('=== Integrality Gap ===')
print(f'LP optimal (upper bound): ${z_LP:.4f}M')
print(f'IP optimal:               ${z_IP:.4f}M')
print(f'Integrality gap:          {gap:.2f}%')
print()
if gap < 5:
    print('Interpretation: TIGHT relaxation (gap < 5%).')
    print('The LP relaxation is a very close upper bound on the IP optimal.')
elif gap < 15:
    print('Interpretation: MODERATE gap. The LP relaxation is reasonably tight.')
else:
    print('Interpretation: LOOSE relaxation. LP significantly overestimates IP.')


=== Integrality Gap ===
LP optimal (upper bound): $57.0000M
IP optimal:               $47.0000M
Integrality gap:          21.28%

Interpretation: LOOSE relaxation. LP significantly overestimates IP.



## Problem 2: Fixed-Charge Transportation

A manufacturer ships products from 3 plants to 5 retail stores. Shipping on any route incurs a fixed cost (for arranging transport) plus a variable per-unit cost. Each plant has a supply limit, and each store has a demand.

| | Store 1 | Store 2 | Store 3 | Store 4 | Store 5 | Supply |
|---|---------|---------|---------|---------|---------|--------|
| **Plant A** | 4 | 8 | 1 | 5 | 7 | 120 |
| **Plant B** | 7 | 3 | 6 | 2 | 4 | 150 |
| **Plant C** | 5 | 6 | 3 | 7 | 2 | 100 |
| **Demand** | 80 | 60 | 70 | 40 | 50 | |

*Variable costs shown above (\$/unit). Fixed costs:*

| | Store 1 | Store 2 | Store 3 | Store 4 | Store 5 |
|---|---------|---------|---------|---------|---------|
| **Plant A** | 200 | 350 | 150 | 250 | 300 |
| **Plant B** | 300 | 180 | 280 | 120 | 200 |
| **Plant C** | 250 | 270 | 160 | 320 | 150 |

### Tasks

**(a)** Formulate a MILP: continuous flow variables $x_{ij}$, binary route-open variables $y_{ij}$, switching constraints linking them. Write `.mod` and `.dat` files.

**(b)** Solve both the LP relaxation and the IP. Report the integrality gap.

In [10]:
# =============================================
# PROBLEM 2: Your .mod file
# =============================================

%%writefile transport_fc.mod
# Fixed-Charge Transportation Model

set PLANTS;
set STORES;

param vcost  {PLANTS, STORES} >= 0;   # variable cost per unit shipped ($)
param fcost  {PLANTS, STORES} >= 0;   # fixed cost to open route ($)
param supply {PLANTS}         >= 0;   # supply capacity (units)
param demand {STORES}         >= 0;   # store demand (units)

# Big-M: largest possible flow on a single route
param M := max {i in PLANTS} supply[i];

var x {PLANTS, STORES} >= 0;           # flow on route i->j
var y {PLANTS, STORES} binary;         # 1 = route i->j is open

minimize TotalCost:
    sum {i in PLANTS, j in STORES} (vcost[i,j] * x[i,j] + fcost[i,j] * y[i,j]);

subject to SupplyLimit {i in PLANTS}:
    sum {j in STORES} x[i,j] <= supply[i];

subject to DemandMet {j in STORES}:
    sum {i in PLANTS} x[i,j] >= demand[j];

# Big-M: flow only if route is open
subject to Linking {i in PLANTS, j in STORES}:
    x[i,j] <= M * y[i,j];


Writing transport_fc.mod


In [11]:
# =============================================
# PROBLEM 2: Your .dat file
# =============================================

%%writefile transport_fc.dat
# Data for Fixed-Charge Transportation

set PLANTS := A B C;
set STORES := S1 S2 S3 S4 S5;

param vcost:
       S1  S2  S3  S4  S5 :=
  A     4   8   1   5   7
  B     7   3   6   2   4
  C     5   6   3   7   2 ;

param fcost:
       S1   S2   S3   S4   S5 :=
  A   200  350  150  250  300
  B   300  180  280  120  200
  C   250  270  160  320  150 ;

param supply :=
  A 120
  B 150
  C 100 ;

param demand :=
  S1 80
  S2 60
  S3 70
  S4 40
  S5 50 ;


Writing transport_fc.dat


In [12]:
# =============================================
# PROBLEM 2(a)-(b): Solve and analyze
# =============================================

fc = AMPL()
fc.read('transport_fc.mod')
fc.read_data('transport_fc.dat')
fc.option['solver'] = 'gurobi'
fc.solve()

total_cost = fc.get_objective('TotalCost').value()
plants = sorted(fc.get_set('PLANTS').members())
stores = sorted(fc.get_set('STORES').members())

x_val = {(i,j): fc.get_variable('x')[i,j].value() for i in plants for j in stores}
y_val = {(i,j): fc.get_variable('y')[i,j].value() for i in plants for j in stores}

print('=== Fixed-Charge Transportation Solution ===')
print(f'Total cost: ${total_cost:,.2f}\n')

print('Open routes and flows:')
for i in plants:
    for j in stores:
        if y_val[(i,j)] > 0.5:
            print(f'  Plant {i} -> {j}: {x_val[(i,j)]:.1f} units')

print('\nFlow matrix (units):')
print('         ' + '  '.join(f'{s:>6}' for s in stores))
for i in plants:
    row = f'Plant {i}:  ' + '  '.join(f'{x_val[(i,j)]:>6.1f}' for j in stores)
    print(row)


Gurobi 13.0.0: Gurobi 13.0.0: optimal solution; objective 1770
22 simplex iterations
1 branching node
=== Fixed-Charge Transportation Solution ===
Total cost: $1,770.00

Open routes and flows:
  Plant A -> S1: 80.0 units
  Plant A -> S3: 40.0 units
  Plant B -> S2: 60.0 units
  Plant B -> S4: 40.0 units
  Plant C -> S3: 30.0 units
  Plant C -> S5: 50.0 units

Flow matrix (units):
             S1      S2      S3      S4      S5
Plant A:    80.0     0.0    40.0     0.0     0.0
Plant B:     0.0    60.0     0.0    40.0     0.0
Plant C:     0.0     0.0    30.0     0.0    50.0


---

## Problem 3: Set Covering with Budget and Performance Tiers

A university is designing a new cybersecurity monitoring system. There are 12 types of threats that must be detected, and 8 candidate detection tools are available. Each tool covers a subset of threats, has a cost, and has a reliability rating.

| Tool | Cost (\$K) | Reliability | Threats Covered |
|------|-----------|-------------|-----------------|
| T1   | 45        | High        | 1, 2, 5, 8      |
| T2   | 30        | Medium      | 2, 3, 6         |
| T3   | 55        | High        | 3, 4, 7, 9, 12  |
| T4   | 25        | Low         | 1, 5, 10        |
| T5   | 40        | Medium      | 4, 6, 8, 11     |
| T6   | 60        | High        | 7, 9, 10, 11, 12|
| T7   | 35        | Medium      | 2, 5, 8, 10     |
| T8   | 50        | High        | 1, 3, 6, 9, 11  |

### Tasks

**(a)** **Minimum-cost covering:** Find the cheapest set of tools that covers all 12 threats. Write the `.mod` and `.dat` files.

**(b)** **Budget-constrained max coverage:** With a budget of \$120K, maximize the number of threats covered. (Hint: add a binary "uncovered" variable $u_i$ for each threat; penalize uncovered threats.)

**(c)** **Redundancy requirement:** Modify model (a) so that every threat is covered by at least **two** tools. How does the cost change?

**(d)** **Reliability tiers:** Define High = 3, Medium = 2, Low = 1. Add a constraint requiring average reliability of selected tools $\geq$ 2.5. Solve the minimum-cost covering model with this added constraint.


In [22]:
# =============================================
# PROBLEM 3: Your .mod and .dat files
# =============================================

%%writefile cybersecurity.mod
# Cybersecurity Threat Coverage Model

set TOOLS;
set THREATS;

param cost    {TOOLS}          >= 0;   # cost in $K
param covers  {TOOLS, THREATS} binary; # 1 if tool covers threat
param rel     {TOOLS}          >= 0;   # reliability (High=3, Med=2, Low=1)
param budget  default 1e9;             # optional budget cap
param min_cov default 1;               # min times each threat must be covered
param min_rel default 0;               # min average reliability (0 = no constraint)

var z {TOOLS} binary;                   # 1 = select this tool

minimize TotalCost:
    sum {t in TOOLS} cost[t] * z[t];

subject to Coverage {i in THREATS}:
    sum {t in TOOLS} covers[t,i] * z[t] >= min_cov;

subject to BudgetLimit:
    sum {t in TOOLS} cost[t] * z[t] <= budget;

# Average reliability >= min_rel
# Equivalent: sum(rel[t]*z[t]) >= min_rel * sum(z[t])
#          => sum((rel[t] - min_rel)*z[t]) >= 0
subject to AvgReliability:
    sum {t in TOOLS} (rel[t] - min_rel) * z[t] >= 0;


Overwriting cybersecurity.mod


In [20]:
%%writefile cybersecurity.dat
# Data for Cybersecurity Model

set TOOLS   := T1 T2 T3 T4 T5 T6 T7 T8;
set THREATS := 1 2 3 4 5 6 7 8 9 10 11 12;

param cost :=
  T1 45  T2 30  T3 55  T4 25
  T5 40  T6 60  T7 35  T8 50 ;

# rel: High=3, Medium=2, Low=1
param rel :=
  T1 3  T2 2  T3 3  T4 1
  T5 2  T6 3  T7 2  T8 3 ;

# covers[tool, threat]: 1 = tool covers that threat
param covers:
       1  2  3  4  5  6  7  8  9 10 11 12 :=
  T1   1  1  0  0  1  0  0  1  0  0  0  0
  T2   0  1  1  0  0  1  0  0  0  0  0  0
  T3   0  0  1  1  0  0  1  0  1  0  0  1
  T4   1  0  0  0  1  0  0  0  0  1  0  0
  T5   0  0  0  1  0  1  0  1  0  0  1  0
  T6   0  0  0  0  0  0  1  0  1  1  1  1
  T7   0  1  0  0  1  0  0  1  0  1  0  0
  T8   1  0  1  0  0  1  0  0  1  0  1  0 ;


Overwriting cybersecurity.dat


In [23]:
# =============================================
# PROBLEM 3(a)-(d): Solve and analyze
# =============================================

def solve_cyber(min_cov=1, budget=1e9, min_rel=0, label=''):
    m = AMPL()
    m.read('cybersecurity.mod')
    m.read_data('cybersecurity.dat')
    m.param['min_cov'] = min_cov
    m.param['budget']  = budget
    m.param['min_rel'] = min_rel
    m.option['solver'] = 'gurobi'
    m.option['gurobi_options'] = 'outlev=0'
    m.solve()
    cost_val = m.get_objective('TotalCost').value()
    selected = [t for t in sorted(m.get_set('TOOLS').members())
                if m.get_variable('z')[t].value() > 0.5]
    print(f'--- {label} ---')
    print(f'  Cost: ${cost_val}K   |   Selected tools: {selected}')
    return cost_val, selected


# (a) Minimum-cost covering (all 12 threats, single coverage)
cost_a, tools_a = solve_cyber(min_cov=1, label='(a) Min-cost covering')

# (b) Budget $120K, maximize number of threats covered
m_b = AMPL()
m_b.eval('''
set TOOLS;  set THREATS;
param cost   {TOOLS}          >= 0;
param covers {TOOLS, THREATS} binary;
param rel    {TOOLS}          >= 0;   # declared to accept full .dat file
param B := 120;
var z       {TOOLS}   binary;
var covered {THREATS} binary;
maximize ThreatsCovered: sum {i in THREATS} covered[i];
subject to BudgetLim:  sum {t in TOOLS} cost[t]*z[t] <= B;
subject to CovDef {i in THREATS}:
    covered[i] <= sum {t in TOOLS} covers[t,i]*z[t];
''')
m_b.read_data('cybersecurity.dat')
m_b.option['solver'] = 'gurobi'
m_b.option['gurobi_options'] = 'outlev=0'
m_b.solve()
threats_cov = int(m_b.get_objective('ThreatsCovered').value())
tools_b = [t for t in sorted(m_b.get_set('TOOLS').members())
           if m_b.get_variable('z')[t].value() > 0.5]
spend_b  = sum(m_b.param['cost'][t] * m_b.get_variable('z')[t].value()
               for t in m_b.get_set('TOOLS').members())
print(f'--- (b) Budget $120K max coverage ---')
print(f'  Threats covered: {threats_cov}/12  |  Budget used: ${spend_b}K  |  Tools: {tools_b}')

# (c) Double coverage (each threat covered by >= 2 tools)
cost_c, tools_c = solve_cyber(min_cov=2, label='(c) Double coverage (min_cov=2)')
print(f'  Cost increase vs (a): +${cost_c - cost_a}K  ({(cost_c-cost_a)/cost_a*100:.1f}%)')

# (d) Min average reliability >= 2.5
cost_d, tools_d = solve_cyber(min_cov=1, min_rel=2.5,
                              label='(d) Min avg reliability >= 2.5')


Gurobi 13.0.0:   tech:outlev = 0
Gurobi 13.0.0: optimal solution; objective 140
7 simplex iterations
1 branching node
--- (a) Min-cost covering ---
  Cost: $140.0K   |   Selected tools: ['T3', 'T7', 'T8']
Gurobi 13.0.0:   tech:outlev = 0
Gurobi 13.0.0: optimal solution; objective 11
16 simplex iterations
1 branching node
--- (b) Budget $120K max coverage ---
  Threats covered: 11/12  |  Budget used: $120.0K  |  Tools: ['T3', 'T4', 'T5']
Gurobi 13.0.0:   tech:outlev = 0
Gurobi 13.0.0: optimal solution; objective 255
0 simplex iterations
1 branching node
--- (c) Double coverage (min_cov=2) ---
  Cost: $255.0K   |   Selected tools: ['T1', 'T2', 'T3', 'T4', 'T5', 'T6']
  Cost increase vs (a): +$115.0K  (82.1%)
Gurobi 13.0.0:   tech:outlev = 0
Gurobi 13.0.0: optimal solution; objective 140
9 simplex iterations
1 branching node
--- (d) Min avg reliability >= 2.5 ---
  Cost: $140.0K   |   Selected tools: ['T3', 'T7', 'T8']




## Problem 4: Emergency Shelter Location After a Natural Disaster

After a major flood, a regional emergency agency must set up temporary shelters. There are 15 affected communities and 8 candidate shelter sites. Each shelter has a capacity and an opening cost. Each community has a displaced population. Travel time (in minutes) from each community to each shelter is given.

**Objective:** Minimize total cost = fixed shelter costs + a penalty of \$100 per person-minute of travel.

**Data:**

| Community | Population | | Shelter | Capacity | Fixed Cost (\$K) |
|-----------|------------|-|---------|----------|-----------------|
| A1 | 200 | | S1 | 500 | 80 |
| A2 | 150 | | S2 | 400 | 65 |
| A3 | 300 | | S3 | 600 | 95 |
| A4 | 180 | | S4 | 350 | 55 |
| A5 | 250 | | S5 | 450 | 70 |
| A6 | 120 | | S6 | 550 | 90 |
| A7 | 280 | | S7 | 300 | 50 |
| A8 | 160 | | S8 | 500 | 75 |
| A9 | 350 | | | | |
| A10 | 90 | | | | |
| A11 | 220 | | | | |
| A12 | 170 | | | | |
| A13 | 130 | | | | |
| A14 | 260 | | | | |
| A15 | 190 | | | | |

*Travel times (minutes):*

| | S1 | S2 | S3 | S4 | S5 | S6 | S7 | S8 |
|---|----|----|----|----|----|----|----|----|
| A1 | 10 | 25 | 30 | 45 | 20 | 35 | 50 | 15 |
| A2 | 30 | 10 | 20 | 35 | 40 | 15 | 25 | 45 |
| A3 | 20 | 35 | 10 | 15 | 30 | 40 | 20 | 25 |
| A4 | 40 | 15 | 25 | 10 | 35 | 20 | 30 | 50 |
| A5 | 15 | 40 | 35 | 30 | 10 | 25 | 45 | 20 |
| A6 | 35 | 20 | 15 | 40 | 25 | 10 | 30 | 35 |
| A7 | 25 | 30 | 40 | 20 | 15 | 35 | 10 | 30 |
| A8 | 45 | 25 | 20 | 30 | 35 | 15 | 40 | 10 |
| A9 | 20 | 15 | 30 | 25 | 40 | 20 | 35 | 30 |
| A10 | 30 | 40 | 15 | 20 | 25 | 30 | 15 | 40 |
| A11 | 15 | 35 | 25 | 30 | 20 | 40 | 25 | 20 |
| A12 | 40 | 10 | 35 | 25 | 30 | 20 | 35 | 25 |
| A13 | 25 | 20 | 40 | 35 | 15 | 25 | 30 | 15 |
| A14 | 35 | 30 | 10 | 15 | 25 | 30 | 40 | 35 |
| A15 | 10 | 25 | 20 | 40 | 35 | 15 | 20 | 30 |

### Tasks

**(a)** Formulate as a MILP with binary shelter-opening variables and continuous assignment variables (allow demand splitting). Write `.mod` and `.dat` files.

**(b)** Solve and report: which shelters open, how are communities assigned, what is the total cost?

**(c)** Add an **equity constraint**: no community should travel more than 30 minutes. Which assignment variables must be eliminated or constrained? Re-solve — how does the cost and solution change?

**(d)** Add a **redundancy constraint**: each community must be reachable by at least 2 open shelters within 40 minutes. Re-solve and discuss the impact on cost.

In [37]:
# =============================================
# PROBLEM 4: Your .mod and .dat files
# =============================================

%%writefile shelter.mod
# Emergency Shelter Location Model

set COMM;          # affected communities
set SHELT;         # candidate shelter sites

param pop      {COMM}        >= 0;   # displaced population per community
param cap      {SHELT}       >= 0;   # shelter capacity
param fcost    {SHELT}       >= 0;   # fixed opening cost ()
param travel   {COMM, SHELT} >= 0;   # travel time (minutes)
param penalty  default 100;          # $ per person-minute
param max_travel default 1e6;        # equity: max allowed travel (minutes); 1e6 = no limit
param min_reach  default 0;          # redundancy: min open shelters within reach_lim minutes
param reach_lim  default 40;         # time window for redundancy check

var y {SHELT} binary;                # 1 = open shelter
var x {COMM, SHELT} >= 0, <= 1;     # fraction of community c assigned to shelter s

# Total cost = fixed costs ( -> $) + penalty * person-minutes
minimize TotalCost:
    1000 * sum {s in SHELT} fcost[s] * y[s]
    + penalty * sum {c in COMM, s in SHELT}
          pop[c] * travel[c,s] * x[c,s];

# All displaced people in each community must be assigned
subject to FullAssign {c in COMM}:
    sum {s in SHELT} x[c,s] = 1;

# Shelter capacity: assign only to open shelters
subject to CapacityLink {s in SHELT}:
    sum {c in COMM} pop[c] * x[c,s] <= cap[s] * y[s];

# Equity: ban routes longer than max_travel (if-expression evaluated at solve time)
subject to EquityBan {c in COMM, s in SHELT}:
    x[c,s] <= if travel[c,s] <= max_travel then 1 else 0;

# Redundancy: each community within reach_lim minutes of >= min_reach open shelters
subject to Redundancy {c in COMM}:
    sum {s in SHELT: travel[c,s] <= reach_lim} y[s] >= min_reach;


Overwriting shelter.mod


In [38]:
%%writefile shelter.dat
# Data for Emergency Shelter Location

set COMM  := A1 A2 A3 A4 A5 A6 A7 A8 A9 A10 A11 A12 A13 A14 A15;
set SHELT := S1 S2 S3 S4 S5 S6 S7 S8;

param pop :=
  A1  200  A2  150  A3  300  A4  180  A5  250
  A6  120  A7  280  A8  160  A9  350  A10  90
  A11 220  A12 170  A13 130  A14 260  A15 190 ;

param cap :=
  S1 500  S2 400  S3 600  S4 350
  S5 450  S6 550  S7 300  S8 500 ;

param fcost :=
  S1 80  S2 65  S3 95  S4 55
  S5 70  S6 90  S7 50  S8 75 ;

param travel:
         S1  S2  S3  S4  S5  S6  S7  S8 :=
  A1     10  25  30  45  20  35  50  15
  A2     30  10  20  35  40  15  25  45
  A3     20  35  10  15  30  40  20  25
  A4     40  15  25  10  35  20  30  50
  A5     15  40  35  30  10  25  45  20
  A6     35  20  15  40  25  10  30  35
  A7     25  30  40  20  15  35  10  30
  A8     45  25  20  30  35  15  40  10
  A9     20  15  30  25  40  20  35  30
  A10    30  40  15  20  25  30  15  40
  A11    15  35  25  30  20  40  25  20
  A12    40  10  35  25  30  20  35  25
  A13    25  20  40  35  15  25  30  15
  A14    35  30  10  15  25  30  40  35
  A15    10  25  20  40  35  15  20  30 ;


Overwriting shelter.dat


In [39]:
# =============================================
# PROBLEM 4(a)-(d): Solve and analyze
# =============================================

def solve_shelter(max_travel=1e6, min_reach=0, reach_lim=40, penalty=100, label=''):
    m = AMPL()
    m.read('shelter.mod')
    m.read_data('shelter.dat')
    m.param['max_travel'] = max_travel
    m.param['min_reach']  = min_reach
    m.param['reach_lim']  = reach_lim
    m.param['penalty']    = penalty
    m.option['solver'] = 'gurobi'
    m.option['gurobi_options'] = 'outlev=0'
    m.solve()

    result = m.solve_result
    shelts = sorted(m.get_set('SHELT').members())
    comms  = sorted(m.get_set('COMM').members())

    print(f'--- {label} ---')
    if result not in ('solved', 'solved?'):
        print(f'  Solver result: {result}  (infeasible or unbounded)')
        return None, None, None

    total  = m.get_objective('TotalCost').value()
    opened = [s for s in shelts if m.get_variable('y')[s].value() > 0.5]
    print(f'  Total cost:     ${total:,.0f}')
    print(f'  Open shelters:  {opened}  ({len(opened)} of 8)')

    xv   = m.get_variable('x')
    trav = m.param['travel']
    assignments = {}
    print('  Community primary assignments:')
    for c in comms:
        best_s, best_v = None, -1.0
        for s in shelts:
            try:
                v = xv[c, s].value()
                if v > best_v:
                    best_v, best_s = v, s
            except:
                pass
        t = int(trav[c, best_s]) if best_s else '?'
        assignments[c] = (best_s, best_v, t)
        print(f'    {c:>3} -> {best_s}  ({best_v*100:.0f}%,  {t} min)')
    return total, opened, assignments


# -------------------------------------------------------
# (b) Base model
# -------------------------------------------------------
cost_base, open_base, assign_base = solve_shelter(
    label='(b) Base model — penalty=$100/person-min')

print()
print('--- (c) Equity Analysis: max 30-min travel ---')
# Identify which routes are blocked by the 30-min cap
travel_data = {
    ('A1','S1'):10,('A1','S2'):25,('A1','S3'):30,('A1','S4'):45,('A1','S5'):20,('A1','S6'):35,('A1','S7'):50,('A1','S8'):15,
    ('A2','S1'):30,('A2','S2'):10,('A2','S3'):20,('A2','S4'):35,('A2','S5'):40,('A2','S6'):15,('A2','S7'):25,('A2','S8'):45,
    ('A3','S1'):20,('A3','S2'):35,('A3','S3'):10,('A3','S4'):15,('A3','S5'):30,('A3','S6'):40,('A3','S7'):20,('A3','S8'):25,
    ('A4','S1'):40,('A4','S2'):15,('A4','S3'):25,('A4','S4'):10,('A4','S5'):35,('A4','S6'):20,('A4','S7'):30,('A4','S8'):50,
    ('A5','S1'):15,('A5','S2'):40,('A5','S3'):35,('A5','S4'):30,('A5','S5'):10,('A5','S6'):25,('A5','S7'):45,('A5','S8'):20,
    ('A6','S1'):35,('A6','S2'):20,('A6','S3'):15,('A6','S4'):40,('A6','S5'):25,('A6','S6'):10,('A6','S7'):30,('A6','S8'):35,
    ('A7','S1'):25,('A7','S2'):30,('A7','S3'):40,('A7','S4'):20,('A7','S5'):15,('A7','S6'):35,('A7','S7'):10,('A7','S8'):30,
    ('A8','S1'):45,('A8','S2'):25,('A8','S3'):20,('A8','S4'):30,('A8','S5'):35,('A8','S6'):15,('A8','S7'):40,('A8','S8'):10,
    ('A9','S1'):20,('A9','S2'):15,('A9','S3'):30,('A9','S4'):25,('A9','S5'):40,('A9','S6'):20,('A9','S7'):35,('A9','S8'):30,
    ('A10','S1'):30,('A10','S2'):40,('A10','S3'):15,('A10','S4'):20,('A10','S5'):25,('A10','S6'):30,('A10','S7'):15,('A10','S8'):40,
    ('A11','S1'):15,('A11','S2'):35,('A11','S3'):25,('A11','S4'):30,('A11','S5'):20,('A11','S6'):40,('A11','S7'):25,('A11','S8'):20,
    ('A12','S1'):40,('A12','S2'):10,('A12','S3'):35,('A12','S4'):25,('A12','S5'):30,('A12','S6'):20,('A12','S7'):35,('A12','S8'):25,
    ('A13','S1'):25,('A13','S2'):20,('A13','S3'):40,('A13','S4'):35,('A13','S5'):15,('A13','S6'):25,('A13','S7'):30,('A13','S8'):15,
    ('A14','S1'):35,('A14','S2'):30,('A14','S3'):10,('A14','S4'):15,('A14','S5'):25,('A14','S6'):30,('A14','S7'):40,('A14','S8'):35,
    ('A15','S1'):10,('A15','S2'):25,('A15','S3'):20,('A15','S4'):40,('A15','S5'):35,('A15','S6'):15,('A15','S7'):20,('A15','S8'):30,
}
comms_list = ['A1','A2','A3','A4','A5','A6','A7','A8','A9','A10','A11','A12','A13','A14','A15']
shelts_list = ['S1','S2','S3','S4','S5','S6','S7','S8']
blocked = [(c, s, t) for (c,s),t in travel_data.items() if t > 30]
print(f'  Routes blocked by equity (travel > 30 min): {len(blocked)} routes')
for c, s, t in sorted(blocked):
    print(f'    x[{c},{s}] = 0  forced  (travel={t} min)')

# Check base assignments: are any > 30 min?
if assign_base:
    violations = [(c, s, t) for c,(s,v,t) in assign_base.items() if t > 30]
    if violations:
        print(f'  Base assignments violating equity: {violations}')
    else:
        print('  Base assignments: all primary assignments are <= 30 min already.')
        print('  => Equity constraint is NON-BINDING: the base optimal solution')
        print('     already routes every community within 30 minutes because the')
        print('     high penalty ($100/person-min) makes distant travel very costly.')

cost_eq, open_eq, _ = solve_shelter(max_travel=30,
                                    label='(c) Equity enforced — max 30-min travel')
if cost_eq is not None and cost_base is not None:
    delta = cost_eq - cost_base
    print(f'  Cost change: ${delta:,.0f}  ({delta/cost_base*100:.2f}%)')
    if delta == 0:
        print('  Cost unchanged: the 30-min cap eliminates only routes that were')
        print('  already unused in the optimal base solution (the penalty makes')
        print('  long routes too expensive to use regardless of the hard cap).')

print()
print('--- (d) Redundancy Analysis: >=2 open shelters within 40 min ---')
# Check redundancy in base: for each community, how many open shelters within 40 min?
if open_base:
    print('  Reachable open shelters within 40 min (base solution):')
    for c in comms_list:
        within40 = [s for s in open_base if travel_data[(c,s)] <= 40]
        flag = '  <-- < 2 !' if len(within40) < 2 else ''
        print(f'    {c:>3}: {within40}  ({len(within40)} shelters){flag}')
    min_reach_base = min(len([s for s in open_base if travel_data[(c,s)] <= 40])
                        for c in comms_list)
    if min_reach_base >= 2:
        print(f'  All communities have >= 2 open shelters within 40 min (min={min_reach_base}).')
        print('  => Redundancy constraint is NON-BINDING: with all/most shelters open,')
        print('     the dense travel network already guarantees >= 2 nearby shelters.')

cost_red, open_red, _ = solve_shelter(min_reach=2, reach_lim=40,
                                      label='(d) Redundancy enforced — >=2 within 40 min')
if cost_red is not None and cost_base is not None:
    delta = cost_red - cost_base
    print(f'  Cost change: ${delta:,.0f}  ({delta/cost_base*100:.2f}%)')
    if delta == 0:
        print('  Cost unchanged: the base solution already satisfies redundancy.')


Gurobi 13.0.0:   tech:outlev = 0
Gurobi 13.0.0: optimal solution; objective 4230000
16 simplex iterations
1 branching node
--- (b) Base model — penalty=$100/person-min ---
  Total cost:     $4,230,000
  Open shelters:  ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8']  (8 of 8)
  Community primary assignments:
     A1 -> S1  (100%,  10 min)
    A10 -> S3  (78%,  15 min)
    A11 -> S1  (50%,  15 min)
    A12 -> S2  (100%,  10 min)
    A13 -> S5  (100%,  15 min)
    A14 -> S3  (100%,  10 min)
    A15 -> S1  (100%,  10 min)
     A2 -> S6  (100%,  15 min)
     A3 -> S3  (90%,  10 min)
     A4 -> S4  (100%,  10 min)
     A5 -> S5  (100%,  10 min)
     A6 -> S6  (100%,  10 min)
     A7 -> S7  (100%,  10 min)
     A8 -> S8  (100%,  10 min)
     A9 -> S2  (66%,  15 min)

--- (c) Equity Analysis: max 30-min travel ---
  Routes blocked by equity (travel > 30 min): 38 routes
    x[A1,S4] = 0  forced  (travel=45 min)
    x[A1,S6] = 0  forced  (travel=35 min)
    x[A1,S7] = 0  forced  (travel=50 min

---

## Appendix: Quick Reference

### AMPL Integer Programming Syntax

```
# Binary variable
var x {SET} binary;

# General integer variable
var n {SET} integer, >= 0;

# Redeclare to relax integrality (for LP relaxation)
redeclare var x {SET} >= 0, <= 1;

# Switching constraint
subject to link {j in SET}: x[j] <= upper_bound[j] * y[j];
```

### Gurobi Options in AMPL

```
option solver gurobi;
option gurobi_options 'outlev=1 mipgap=0.01 timelim=300 cuts=2 presolve=2';
```

| Option | Values | Description |
|:-------|:-------|:------------|
| `outlev` | 0, 1 | Silent or verbose log |
| `mipgap` | 0.0–1.0 | Relative optimality gap tolerance |
| `timelim` | seconds | Max solve time |
| `cuts` | 0, 1, 2, 3 | Cut aggressiveness |
| `presolve` | 0, 1, 2 | Presolve level |
| `heuristics` | 0.0–1.0 | Fraction of effort on heuristics |
| `threads` | integer | Number of parallel threads |

### Integrality Gap

$$\text{Gap} = \frac{|z^*_{IP} - z^*_{LP}|}{|z^*_{IP}|} \times 100\%$$

- **Minimization:** $z^*_{LP} \leq z^*_{IP}$ (LP gives lower bound)
- **Maximization:** $z^*_{LP} \geq z^*_{IP}$ (LP gives upper bound)
